# <span style="color:darkblue"> **Project 1: Social Network Jornal Web Scraping**  </span>

<font size = "5"> 

Overview

<font size = "3"> 

In this project, I scraped the most recent five years of articles from the journal **Social Networks** on **ScienceDirect**. To achieve this, I used webdriver to dynamically load the journal’s yearly panels, collected volume metadata, and visited each issue links to extract article level data, **title**, **article type**, and **article url**. For each article, I then visited the detail page to gather **authors** and **keywords** for future analysis.

In the procedure, I found out the site blocked automated requests for articles after scraping for around 90 seconds. To solve this, I implemented a **resumable web scraping**: I recorded the progress of scraping in a json file and added every articles to csv after it is processed, and updated the progress in json and proceed to the next article. I terminated the scraping once the csv has covered the latest five years' publications.

<font size = "5"> 

Datasource

<font size = "3"> 

Data was collected from the journal *Social Networks* on ScienceDirect, which publishes empirical research in network science and social structure analysis, and critical reviews of approaches and books. 

- **Main journal page**: https://www.sciencedirect.com/journal/social-networks/issues

- **Example volume page (Volume 84)**: https://www.sciencedirect.com/journal/social-networks/vol/84/suppl/C

- **Example article page (first article in Volume 84)**: https://www.sciencedirect.com/science/article/pii/S0378873325000498

Data was collected between 10/17/2025-10/20/2025


<font size = "5"> 

Data
<font size = "3">

Following data was collected:
| Variable | Description |
|-------|--------------|
| **Volume** | Volume number |
| **Date** | Publication date |
| **Article** | Article title |
| **Authors** | List of authors (combined given names and surnames) |
| **Article_URL** | Direct link to the article page |
| **Type** | Article type (e.g., Research Article, Book Review) |
| **Keywords** | Keywords listed in the article |

# <span style="color:darkblue"> **Step 1: Use Webdriver to Open Jornal Website**  </span>

<font size = "4"> 

First, I imported the packages and defined the journal url, and opened it with Webdriver.


In [25]:
# Import packages
import requests
import json
import jmespath
import pandas as pd
from pandas import DataFrame
import matplotlib.pyplot as plt
from collections import Counter
from bs4 import BeautifulSoup
import time
import csv
import random
import os, re
exec(open("/Users/zhangxiaobei/Documents/GitHub/datasci530fall2025/Lecture 06/scripts/import_packages.py").read())

In [26]:
# Install webdriver
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

In [27]:
options = webdriver.ChromeOptions()

In [28]:
# Open the Social Network Journal website
url = "https://www.sciencedirect.com/journal/social-networks/issues"
driver.get(url)

# <span style="color:darkblue"> **Step 2: Scrape Journal Volume**  </span>

<font size = "4"> 

In this section, I finished the first stage of the web scraping pipeline to collect journal volume links, publication dates, and issue identifiers from the *Social Networks* journal’s main page. As I focused on the last 5 years' publications, I would not proceed to the next page. 

The main journal page organizes volumes by yearly panel, each of which can be expanded to display the issues published in that year. I used xpath to locate all panels and expanded them by clicking buttons.

- I included a condition to check whether the panel was already **expanded**, in order to prevent folding an already expanded panel.

Within each expanded panel, I used xpath to retrieve the volume number, issue dates and url for subsequent scraping of individual articles. I stored the extracted results in lists, they are already correspondent by their orginal indexes. 


In [5]:
#Scrape journal volume  links 
# Find all year panels on the page,
# Each panel include one year's issues
panels = driver.find_elements("xpath", '//li[contains(@class,"accordion-panel")]')

# Prepare empty lists to store volumes names, dates and links
volumes = []
vol_dates = []
links = []
# Loop over each panel and expand the panel
for i, panel in enumerate(panels):
    buttons = driver.find_elements("xpath", '//button[contains(@class,"accordion-panel-title")]')
    button = buttons[i]
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
    time.sleep(0.5)
    # Check if the panel is already expanded
    expanded = button.get_attribute("aria-expanded")
    if expanded != "true": 
        try:
            button.click()
        except:
            driver.execute_script("arguments[0].click();", button)
        time.sleep(1.5)
    #Exract volume number, issue dates and issue links
    issue_divs = panel.find_elements("xpath", './/div[contains(@class, "issue-item")]')
    for div in issue_divs:
        try:
            vol = div.find_element("xpath", './/a[contains(@href, "/journal/social-networks/vol/")]')
            vol_date = div.find_element("xpath", './/h3[contains(@class, "js-issue-status")]')

            volume = vol.text.strip()
            date_text = vol_date.text.strip()
            link = vol.get_attribute("href")
            # Save to the lists
            if volume:
                volumes.append(volume)
                vol_dates.append(date_text)
                links.append(link)
        
        except Exception as e:
            print(f"Error at panel {i}: {e}")
            continue
print(volumes)
print(vol_dates)

['Volume 84', 'Volume 83', 'Volume 82', 'Volume 81', 'Volume 80', 'Volume 79', 'Volume 78', 'Volume 77', 'Volume 76', 'Volume 75', 'Volume 74', 'Volume 73', 'Volume 72', 'Volume 71', 'Volume 70', 'Volume 69', 'Volume 68', 'Volume 67', 'Volume 66', 'Volume 65', 'Volume 64', 'Volume 63', 'Volume 62', 'Volume 61', 'Volume 60', 'Volume 59', 'Volume 58', 'Volume 57', 'Volume 56', 'Volume 55', 'Volume 54', 'Volume 53', 'Volume 52', 'Volume 51', 'Volume 50', 'Volume 49', 'Volume 48', 'Volume 47', 'Volume 46', 'Volume 45', 'Volume 44', 'Volume 43', 'Volume 42', 'Volume 41', 'Volume 40', 'Volume 39', 'Volume 38', 'Volume 37', 'Volume 36', 'Volume 35, Issue 4', 'Volume 35, Issue 3', 'Volume 35, Issue 2', 'Volume 35, Issue 1', 'Volume 34, Issue 4', 'Volume 34, Issue 3', 'Volume 34, Issue 2', 'Volume 34, Issue 1', 'Volume 33, Issue 4', 'Volume 33, Issue 3', 'Volume 33, Issue 2', 'Volume 33, Issue 1', 'Volume 32, Issue 4', 'Volume 32, Issue 3', 'Volume 32, Issue 2', 'Volume 32, Issue 1', 'Volume 31

# <span style="color:darkblue"> **Step 3: Scrape Articles**  </span>

<font size = "4"> 

During article scraping from issue pages, access was blocked after several requests. So I implemented resumable web scraping that records progress to a json file and can restart exactly where it stopped.

I iterated through each volume’s article lists pages using the volume links collected from step 2, extracting the article title, article type, and article url. It then visited each article’s detail page to retrieve authors and keywords. After each successful article, one row was appended to the csv and the progress in json was proceeded.

**Coding Rationales**: 

- **Resumable design**: Everytime an article was proccessed, the scraper would write the current`(vol_idx, art_idx)` to json file and the extracted information would be stored in csv. When it finished, the progress json file will move to the next index.

- **Block Detection**: When loading an article url, the scraper checked the page if there was block message. And it would exit the scraping process once it detected blocked message. If not, the access was marked as successful and it would proceed to the article page to retrieve keywords and authors. 

- **Author Extraction**: Extracting authors from the article list pages can be problematic in the case when some articles have multiple authors and the author list can be truncated with "...". To avoid missing authors, the scraper collected given names and surnames from each article’s page and joined them into full names.




In [ ]:
# Resumable web scraping
# Define output files
OUT_CSV = "social_networks_articles.csv"
PROG = "progress.json"

# Read urls that have been written in csv
done_urls = set()
if os.path.exists(OUT_CSV):
    try:
        with open(OUT_CSV, "r", encoding="utf-8") as fr:
            rdr = csv.DictReader(fr)
            # find the article url column
            url_col = None
            for col in rdr.fieldnames or []:
                if col.strip().lower() == "article_url":
                    url_col = col
                    break
            # read the url and add it to done urls set
            if url_col:
                for row in rdr:
                    u = row.get(url_col)
                    if u:
                        done_urls.add(u)
    except Exception as e:
        print("Warning reading existing CSV for done_urls:", e)

# read progress from json file
start_vol = 0
start_art = 0
if os.path.exists(PROG):
    try:
        with open(PROG, "r", encoding="utf-8") as pf:
            prog = json.load(pf)
            start_vol = int(prog.get("vol_idx", 0))
            start_art = int(prog.get("art_idx", 0))
            print("Resuming from progress:", start_vol, start_art)
    except Exception as e:
        print("Warning reading progress.json:", e)

# open csv file for writting
write_mode = "a" if os.path.exists(OUT_CSV) else "w"
with open(OUT_CSV, write_mode, newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    #if first time write csv, define column names
    if write_mode == "w": 
        writer.writerow(["Volume", "Date", "Article", "Authors", "Article_URL", "Type", "Keywords"])
        f.flush()

    # volume loop
    # read links to each volume
    # extract article title, type, and link
    for i in range(start_vol, len(links)):
        link = links[i]
        driver.get(link)
        time.sleep(random.uniform(3, 10))

        articles = driver.find_elements("xpath", '//li[contains(@class, "js-article-list-item")]')
        article_metas = []
        for art in articles:
            #article title
            try:
                title = art.find_element("xpath", './/span[contains(@class, "js-article-title")]').text.strip()
            except:
                title = ""
            # Article type: research articles or others
            try:
                atype = art.find_element("xpath", './/span[contains(@class, "js-article-subtype")]').text.strip()
            except:
                atype = ""
            # article link
            try:
                link_art = art.find_element("xpath", './/a[contains(@href, "/science/article/pii")]')
                url = link_art.get_attribute("href")
            except:
                url = ""
            if url:
                article_metas.append({"title": title, "type": atype, "url": url}) 

        #starts from last time's last article or the new volume's first article
        art_start_index = start_art if i == start_vol else 0

        #Iterate through each article in the current volume
        for j in range(art_start_index, len(article_metas)):
            meta = article_metas[j]
            article_url = meta["url"]

            # Check if the article has been read
            if article_url in done_urls:
                print("  skip already done:", article_url)
                # update progress to next article
                prog = {"vol_idx": i, "art_idx": j+1}
                tmp = PROG + ".tmp"
                with open(tmp, "w", encoding="utf-8") as pf:
                    json.dump(prog, pf)
                os.replace(tmp, PROG)
                continue

            # write progress in json file
            prog = {"vol_idx": i, "art_idx": j}
            tmp = PROG + ".tmp"
            with open(tmp, "w", encoding="utf-8") as pf:
                json.dump(prog, pf)
            os.replace(tmp, PROG)

            # Access article page
            # if blocked, exit the web scrabing 
            success = False
            try:
                driver.get(article_url)
                time.sleep(random.uniform(3, 8))
                page = driver.page_source.lower()
                #detect if the web block
                if "there was a problem providing the content you requested" in page or "access denied" in page:
                    print(f"   Detected block for {article_url}")
                    raise SystemExit("Blocked")
            
                success = True
            # Exit if blocked
            except SystemExit:
                    raise
            except Exception as e:
                    print("   get article error:", e)
                    success = False

            # Extract keywords and authors from articles' pages
            keywords = ""
            if success:
                try:
                    keywords_eles = driver.find_elements("xpath", '//div[@class = "keyword"]')
                    kw = [k.text.strip() for k in keywords_eles if k.text.strip()]
                    keywords = ", ".join(kw)
                except Exception as e:
                    keywords = ""
                
                # Author names are given seperately
                # use join to combine given names and surnames
                try:
                    author = ", ".join(
                        f"{g.text.strip()} {s.text.strip()}".strip()
                        for g, s in zip(
                            driver.find_elements("xpath", '(//*[@id="author-group" or contains(@class,"author-group")])[1]//span[contains(@class,"given-name")]'),
                            driver.find_elements("xpath", '(//*[@id="author-group" or contains(@class,"author-group")])[1]//span[contains(@class,"surname")]')
                        )
                    )
                except:
                    author = ""

            # Write results to csv
            writer.writerow([volumes[i], vol_dates[i], meta["title"], author, article_url, meta["type"], keywords])
            f.flush()
            done_urls.add(article_url)

            # Update progress
            prog = {"vol_idx": i, "art_idx": j+1}
            tmp = PROG + ".tmp"
            with open(tmp, "w", encoding="utf-8") as pf:
                json.dump(prog, pf)
            os.replace(tmp, PROG)

        # Reset start_art after the volume is completed
        start_art = 0

print("Finished (or exited due to block).")


Resuming from progress: 20 19
   Detected block for https://www.sciencedirect.com/science/article/pii/S0378873320300423


SystemExit: Blocked

# <span style="color:darkblue"> **Quality Checks**  </span>

<font size = "4"> 

In this section, I checked the data quality by computing the total number of observations and missing values.

The dataset includes 392 records across 22 volumes.

In [29]:
# load data
networks = pd.read_csv("social_networks_articles.csv")
networks.head()

,Volume,Date,Article,Authors,Article_URL,Type,Keywords
0,Volume 84,In progress (January 2026),Network interventions to improve search and fa...,"Jennifer Watling Neal, Zachary P. Neal",https://www.sciencedirect.com/science/article/...,Research article,"Intervention, Research-practice gap, Research-..."
1,Volume 84,In progress (January 2026),The co-evolution of informal social status and...,"Emily Kruidhof, Rense Corten, Lea Ellwardt, Ra...",https://www.sciencedirect.com/science/article/...,Research article,"Co-evolution, Informal social status, Gossip, ..."
2,Volume 84,In progress (January 2026),Networked inequality: The role of changes in n...,"Alejandro Plaza, Guillermo Beck, Julio Iturra-...",https://www.sciencedirect.com/science/article/...,Research article,"Personal networks, Social influence, Social cl..."
3,Volume 84,In progress (January 2026),Robust network scale-up method estimators,"Sergio Díaz-Aranda, Juan Marcos Ramírez, Jose ...",https://www.sciencedirect.com/science/article/...,Research article,"Network scale-up method, Aggregated relational..."
4,Volume 84,In progress (January 2026),Can an eye for an eye turn the whole world san...,"Khrystyna Holynska, Renato Corbetta, Carter T....",https://www.sciencedirect.com/science/article/...,Research article,"International sanctions, Social network analys..."


In [30]:
# Variables and number of observations
networks.shape

(392, 7)

In [31]:
# number of volumes extracted
networks.nunique()

Volume          22
Date            22
Article        371
Authors        350
Article_URL    392
Type             5
Keywords       145
dtype: int64

In [ ]:
# Number of articles
networks["Type"].value_counts()

Type
Research article       361
Editorial board         21
Erratum                  5
Editorial                4
Short communication      1
Name: count, dtype: int64

In [ ]:
# Display missing values
networks.isna().sum()

Volume           0
Date             0
Article          0
Authors         21
Article_URL      0
Type             0
Keywords       240
dtype: int64

<font size = "4">

**Notes on Missing Values**:

<font size = "4">

- **Authors (21 missing)**

    In each volume, the first entry is the “Editorial Board” page, which does not list authors. I scraped 22 volumes, the Volume 84 has not been completed so it does not have a “Editorial Board” page. Theerefore, we observe 21 missing values in Authors. These rows will be dropped in data cleaning.

- **Keywords(240 missing)**

  By manually checking the data, including visiting article pages, I found that many articles on ScienceDirect do not display keywords when the article is under restricted access. This does not necessarily mean the article lacks keywords, rather the keywords are simply not displayed on the page.


# <span style="color:darkblue"> **Discussion**  </span>

<font size = "4"> 

This project employed dynamic web scraping to collect publication data from the journal Social Networks on ScienceDirect. The main challenge encountered was ScienceDirect’s blocking mechanism, which restricted automated requests after a short period. To address this, I implemented a resumable scraping design that allows repeated web scraping and can be resumed from where it was stopped.

The scraper successfully collected 22 volumes and 392 records, among which 361 are valid research articles. Missing keywords were observed in 240 records, largely due to restricted-access articles that did not publicly display keyword information. These limitations will be considered during data cleaning and subsequent analysis.

During scraping, because the process was not strictly volume based and pauses occurred at random intervals, a few 2020 articles were also included. These articles and non research entries will be dropped in subsequent data cleaning to ensure the dataset covers only the most recent five years publications (Volume 84–64).